In [155]:
import nltk
import pandas as pd
import re
import numpy as np
import pickle
import os
import joblib
from nltk.stem import RSLPStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, recall_score, f1_score, precision_score

In [156]:
# Fazendo o download de palavras que serão excluidas dos textos
nltk.download('words')
nltk.download('stopwords')

[nltk_data] Downloading package words to /home/vitor/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package stopwords to /home/vitor/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [157]:
def salvar_modelo(modelo, vetorizador, label_encoder, diretorio='modelo_nlp'):
    try:
        # Criar o diretório se não existir
        if not os.path.exists(diretorio):
            os.makedirs(diretorio)
        
        # Salvar o modelo
        joblib.dump(modelo, os.path.join(diretorio, 'modelo.pkl'))
        
        # Salvar o vetorizador
        joblib.dump(vetorizador, os.path.join(diretorio, 'vetorizador.pkl'))
        
        # Salvar o label encoder
        joblib.dump(label_encoder, os.path.join(diretorio, 'label_encoder.pkl'))
        
        # Salvar um arquivo de metadados (opcional)
        metadata = {
            'modelo_tipo': type(modelo).__name__,
            'num_caracteristicas': modelo.n_features_in_,
            'classes': list(label_encoder.classes_)
        }
        with open(os.path.join(diretorio, 'metadata.pkl'), 'wb') as f:
            pickle.dump(metadata, f)
            
        print(f"Modelo salvo com sucesso no diretório: {diretorio}")
        return True
    
    except Exception as e:
        print(f"Erro ao salvar o modelo: {str(e)}")
        return False

In [158]:
def carregar_modelo(diretorio='modelo_nlp'):
    try:
        # Verificar se o diretório existe
        if not os.path.exists(diretorio):
            print(f"Diretório não encontrado: {diretorio}")
            return None, None, None
        
        # Carregar o modelo
        modelo_path = os.path.join(diretorio, 'modelo.pkl')
        if not os.path.exists(modelo_path):
            print(f"Arquivo do modelo não encontrado: {modelo_path}")
            return None, None, None
        
        modelo = joblib.load(modelo_path)
        
        # Carregar o vetorizador
        vetorizador_path = os.path.join(diretorio, 'vetorizador.pkl')
        if not os.path.exists(vetorizador_path):
            print(f"Arquivo do vetorizador não encontrado: {vetorizador_path}")
            return None, None, None
            
        vetorizador = joblib.load(vetorizador_path)
        
        # Carregar o label encoder
        label_encoder_path = os.path.join(diretorio, 'label_encoder.pkl')
        if not os.path.exists(label_encoder_path):
            print(f"Arquivo do label encoder não encontrado: {label_encoder_path}")
            return None, None, None
            
        label_encoder = joblib.load(label_encoder_path)
        
        print(f"Modelo carregado com sucesso do diretório: {diretorio}")
        
        # Carregar e exibir metadados (opcional)
        metadata_path = os.path.join(diretorio, 'metadata.pkl')
        if os.path.exists(metadata_path):
            with open(metadata_path, 'rb') as f:
                metadata = pickle.load(f)
            print(f"Tipo do modelo: {metadata['modelo_tipo']}")
            print(f"Número de características: {metadata['num_caracteristicas']}")
            print(f"Classes: {', '.join(metadata['classes'])}")
        
        return modelo, vetorizador, label_encoder
    
    except Exception as e:
        print(f"Erro ao carregar o modelo: {str(e)}")
        return None, None, None

In [159]:
def pre_processar(texto):
    # Lidar com entradas que não são strings
    if not isinstance(texto, str):
        texto = str(texto) if texto is not None else ""
    
    # Converter para minúsculas
    texto = texto.lower()
    
    # Remover URLs
    texto = re.sub(r"http[s]?://\S+", "", texto)
    
    # Remover endereços de email
    texto = re.sub(r"\S+@\S+", "", texto)
    
    # Remover números isolados, mas manter palavras com números
    texto = re.sub(r"\b\d+\b", "", texto)
    
    # Substituir múltiplos espaços por um único espaço
    texto = re.sub(r"\s+", " ", texto)
    
    # Remover a maioria da pontuação (de forma mais abrangente)
    texto = re.sub(r"[^\w\s]", "", texto)
    
    # Remover espaços extras
    texto = texto.strip()
    
    # Obter stopwords do NLTK (em português)
    try:
        stopwords = set(nltk.corpus.stopwords.words('portuguese'))
        # Adicionar stopwords personalizadas, se necessário
        stopwords_personalizadas = {'etc', 'ex', 'ou seja'}
        stopwords.update(stopwords_personalizadas)
    except:
        # Alternativa caso os dados do NLTK não estejam disponíveis
        stopwords = set(['a', 'o', 'e', 'é', 'de', 'da', 'do', 'em', 'que', 'um',
                         'uma', 'os', 'as', 'para', 'com', 'não', 'se', 'na', 'por',
                         'mais', 'as', 'dos', 'como', 'mas', 'ao', 'ele', 'ela', 'foi',
                         'pelo', 'pela', 'até', 'isso', 'ela', 'entre', 'depois',
                         'assim', 'quando', 'mesmo', 'nos', 'já', 'seu', 'sua', 'ou',
                         'ser', 'pelo', 'pela', 'também', 'só'])
    
    # Tokenizar e remover stopwords
    palavras = [palavra for palavra in texto.split() if palavra not in stopwords and len(palavra) > 1]

    # Para português, você pode usar o RSLPStemmer
    stemmer = RSLPStemmer()
    palavras = [stemmer.stem(palavra) for palavra in palavras]
    
    # Rejuntar palavras em uma única string
    texto_processado = " ".join(palavras)
    
    return texto_processado

In [160]:
def prever_categoria(texto, modelo, vetorizador, label_encoder):
    # trata texto igual foi realizado para o treinomento do modelo
    texto_processado = pre_processar(texto)
    # transformar o texto em uma valor numerico
    texto_vetorizado = vetorizador.transform([texto_processado])
    # fazer a previsao
    previsao = modelo.predict(texto_vetorizado)
    # converter a saida do movelo para sua versão de classe em texto
    categoria = label_encoder.inverse_transform(previsao)
    return categoria[0]

In [161]:
# Importando base de dados
df = pd.read_csv('bbc_data.csv')
df.head()

,data,labels
0,Musicians to tackle US red tape Musicians gro...,entertainment
1,"U2s desire to be number one U2, who have won ...",entertainment
2,Rocker Doherty in on-stage fight Rock singer ...,entertainment
3,Snicket tops US box office chart The film ada...,entertainment
4,"Oceans Twelve raids box office Oceans Twelve,...",entertainment


In [162]:
df.shape

(2225, 2)

In [163]:
# Definição de Colunas que serão usadas no Treinamento
class_column = 'labels'
data_column = 'data'

In [164]:
# Quantidade de valores por Classe
df[class_column].value_counts()

labels
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64

In [165]:
df[data_column] = df[data_column].astype(str).fillna('')
# Tratar texto
X = [pre_processar(i) for i in df[data_column]]
# Transformar array de Texto em valores numericos
vetorizador = CountVectorizer(analyzer="word")
X = vetorizador.fit_transform(X)

In [166]:
# Transformar o valor das classes em numeros
le = LabelEncoder()
y = le.fit_transform(df[class_column])


In [167]:
# Separar Base de dados em Treino e Teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X , y , random_state=42 , stratify=y)

In [168]:
# Realizar Treinamento do modelo
modelo = MultinomialNB()
modelo.fit(X_treino , y_treino)

MultinomialNB()

In [169]:
# Realizar previsão com base dos dados de teste
previsoes = modelo.predict(X_teste)

In [170]:
# Avaliando a performance do modelo
acuracia = accuracy_score(y_teste, previsoes)
precision = precision_score(y_teste, previsoes, average='weighted')
precision_classes = precision_score(y_teste, previsoes, average=None)
recall = recall_score(y_teste, previsoes, average='weighted')
f1 = f1_score(y_teste, previsoes, average='weighted')

print(f'Acuracia: {acuracia}, Precisão: {precision}, Precisão por Classe: {precision_classes} Recall: {recall}, F1: {f1}')

Acuracia: 0.9730700179533214, Precisão: 0.973574094436912, Precisão por Classe: [0.98373984 0.9787234  0.95327103 1.         0.94285714] Recall: 0.9730700179533214, F1: 0.973052986769446


In [172]:
# Para salvar o modelo treinado:
salvar_modelo(modelo, vetorizador, le, diretorio='model')

# Para carregar o modelo posteriormente:
modelo_carregado, vetorizador_carregado, le_carregado = carregar_modelo(diretorio='model')

# Teste com um novo texto
texto_novo = "O time de queimada venceu o campeonato nacional pela terceira vez consecutiva."
categoria_prevista = prever_categoria(texto_novo, modelo_carregado, vetorizador_carregado, le_carregado)
print(f"Categoria prevista: {categoria_prevista}")

Modelo salvo com sucesso no diretório: model
Modelo carregado com sucesso do diretório: model
Tipo do modelo: MultinomialNB
Número de características: 27378
Classes: business, entertainment, politics, sport, tech
Categoria prevista: sport
